# 🏭 Synthetic Goldset Generation from Documents

Génère des paires Q/A factuelles **à partir des documents ingérés** (`rag_documents`).

**Objectif** : Créer un goldset avec des questions dont la réponse est **uniquement trouvable dans les documents**, 
pour prouver la valeur ajoutée du RAG vs un LLM seul (mode dry).

**Pipeline** :
1. Charger chaque document markdown depuis `rag_documents`
2. Envoyer au LLM avec un prompt demandant 5-10 Q/A factuelles
3. Stocker dans `goldset_questions_v2` avec `gold_answer`, `gold_sources`, et le `doc_id` source

**LLM** : Scaleway `gpt-oss-120b` (fiable, pas de dépendance Albert)

**Goldset name** : `synthetic_docs_v1`

In [ ]:
# =============================================================================
# Cell 1 — IMPORTS & SETUP
# =============================================================================
import os, sys, json, time, re
from pathlib import Path
from datetime import datetime
from typing import Optional

import pandas as pd
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env')

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"✅ Imports OK")

In [ ]:
# =============================================================================
# Cell 2 — DATABASE CONNECTION
# =============================================================================
from src.rag_v3.config import get_dsn

TUNNEL_DSN = os.getenv("TUNNEL_DSN", "")

def get_db_dsn():
    if TUNNEL_DSN:
        return TUNNEL_DSN
    return get_dsn()

dsn = get_db_dsn()

with psycopg.connect(dsn, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT count(*) as cnt FROM rag_documents WHERE doc_markdown IS NOT NULL')
        doc_count = cur.fetchone()['cnt']
        cur.execute('SELECT count(*) as cnt FROM goldset_questions_v2')
        q_count = cur.fetchone()['cnt']

print(f"✅ DB connected")
print(f"   📄 {doc_count} documents with markdown")
print(f"   ❓ {q_count} existing questions")

In [ ]:
# =============================================================================
# Cell 3 — CONFIGURATION
# =============================================================================

GOLDSET_NAME = "synthetic_docs_v1"   # nom du goldset
QA_PER_DOC = 5                        # questions par document
MAX_DOC_CHARS = 25_000                # tronquer les docs trop longs
LLM_PROVIDER = "scaleway"             # openai | mistral | scaleway
LLM_MODEL = "gpt-oss-120b"            # modèle pour la génération
SAVE_TO_DB = True                      # sauvegarder en DB
SKIP_EXISTING = True                   # ne pas regénérer si déjà fait

print(f"📋 Config:")
print(f"   Goldset: {GOLDSET_NAME}")
print(f"   Q/A per doc: {QA_PER_DOC}")
print(f"   Max doc chars: {MAX_DOC_CHARS}")
print(f"   LLM: {LLM_PROVIDER}/{LLM_MODEL}")
print(f"   Save to DB: {SAVE_TO_DB}")
print(f"   Skip existing: {SKIP_EXISTING}")

In [ ]:
# =============================================================================
# Cell 4 — LLM CLIENT SETUP
# =============================================================================
from openai import OpenAI

def get_llm_client():
    """Initialize LLM client based on provider."""
    if LLM_PROVIDER == "openai":
        return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    elif LLM_PROVIDER == "mistral":
        return OpenAI(
            api_key=os.getenv("MISTRAL_API_KEY"),
            base_url="https://api.mistral.ai/v1"
        )
    elif LLM_PROVIDER == "scaleway":
        return OpenAI(
            api_key=os.getenv("SCALEWAY_API_KEY"),
            base_url=os.getenv("SCALEWAY_BASE_URL", "https://api.scaleway.ai/v1")
        )
    else:
        raise ValueError(f"Unknown provider: {LLM_PROVIDER}")

client = get_llm_client()

# Quick test
test = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "Dis 'OK' si tu fonctionnes."}],
    max_tokens=10
)
print(f"✅ LLM ready: {LLM_PROVIDER}/{LLM_MODEL}")
print(f"   Test: {test.choices[0].message.content}")

In [ ]:
# =============================================================================
# Cell 5 — LOAD DOCUMENTS
# =============================================================================

with psycopg.connect(dsn, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT doc_id::text, short_id, title, publisher, char_count, doc_markdown
            FROM rag_documents
            WHERE doc_markdown IS NOT NULL AND doc_markdown != ''
            ORDER BY publisher, title
        ''')
        documents = cur.fetchall()

print(f"📄 Loaded {len(documents)} documents")
print(f"   MATTE: {sum(1 for d in documents if d['publisher'] == 'MATTE')}")
print(f"   Service-Public: {sum(1 for d in documents if d['publisher'] == 'Service-Public')}")
print(f"\n📏 Size distribution:")
sizes = [d['char_count'] or 0 for d in documents]
print(f"   Min: {min(sizes):,} chars")
print(f"   Median: {sorted(sizes)[len(sizes)//2]:,} chars")
print(f"   Max: {max(sizes):,} chars")
print(f"   > {MAX_DOC_CHARS:,} chars (will be truncated): {sum(1 for s in sizes if s > MAX_DOC_CHARS)}")

In [ ]:
# =============================================================================
# Cell 6 — CHECK EXISTING (skip already generated)
# =============================================================================

existing_short_ids = set()
if SKIP_EXISTING:
    with psycopg.connect(dsn, row_factory=dict_row) as conn:
        with conn.cursor() as cur:
            # We store short_id in gold_sources for traceability
            cur.execute('''
                SELECT DISTINCT gold_sources 
                FROM goldset_questions_v2
                WHERE goldset_name = %s AND gold_sources IS NOT NULL
            ''', (GOLDSET_NAME,))
            existing_short_ids = {r['gold_sources'] for r in cur.fetchall()}
    print(f"⏭️  {len(existing_short_ids)} short_ids already processed — will skip")
else:
    print("🔄 Skip existing: OFF — will regenerate all")

docs_to_process = [d for d in documents if d.get('short_id') not in existing_short_ids]
print(f"📝 {len(docs_to_process)} documents to process")

In [ ]:
# =============================================================================
# Cell 7 — Q/A GENERATION PROMPT & FUNCTION
# =============================================================================

SYSTEM_PROMPT = """Tu es un expert en création de datasets d'évaluation pour des systèmes RAG (Retrieval-Augmented Generation) dans le domaine des ressources humaines de la fonction publique française.

Ton rôle est de générer des paires question/réponse **factuelles et précises** à partir d'un document RH.

RÈGLES CRITIQUES :

1. **Questions spécifiques au document** : Chaque question doit porter sur une information FACTUELLE PRÉCISE contenue dans le document (chiffres, dates, durées, conditions, montants, références juridiques, procédures spécifiques). Un LLM sans accès au document ne devrait PAS pouvoir y répondre correctement.

2. **Pas de questions génériques** : Évite les questions trop larges ("Qu'est-ce que le RIFSEEP ?") ou dont la réponse est de la culture générale RH. Préfère des questions sur des DÉTAILS PRÉCIS du document.

3. **Variété** : Couvre différentes sections du document. Inclus des questions sur :
   - Des montants, seuils ou pourcentages spécifiques
   - Des délais ou durées précis
   - Des conditions d'éligibilité détaillées
   - Des procédures étape par étape
   - Des références à des articles de loi ou décrets spécifiques
   - Des exceptions ou cas particuliers mentionnés

4. **Réponses complètes et sourcées** : Chaque réponse doit :
   - Être factuelle et vérifiable dans le document
   - Inclure les chiffres/dates/références exacts
   - Faire 2-5 phrases (ni trop courte, ni trop longue)
   - Citer la section ou l'article du document si pertinent

5. **Format strict JSON** : Retourne UNIQUEMENT un tableau JSON, sans texte avant ou après."""


def make_user_prompt(doc_title: str, doc_publisher: str, doc_content: str, n_questions: int) -> str:
    """Build the user prompt for Q/A generation."""
    # Truncate if needed
    if len(doc_content) > MAX_DOC_CHARS:
        doc_content = doc_content[:MAX_DOC_CHARS] + "\n\n[... document tronqué ...]"
    
    return f"""Voici un document RH du {doc_publisher} :

**Titre** : {doc_title}

---
{doc_content}
---

Génère exactement {n_questions} paires question/réponse factuelles à partir de ce document.

Retourne un tableau JSON avec ce format exact :
```json
[
  {{
    "question": "La question précise et factuelle",
    "answer": "La réponse complète avec les détails du document (2-5 phrases)",
    "source_section": "La section/partie du document d'où vient l'info"
  }}
]
```

RAPPEL : Les questions doivent être IMPOSSIBLES à répondre correctement sans avoir lu ce document spécifique. 
Cible des détails factuels précis (montants, dates, délais, conditions, articles de loi)."""


def generate_qa_pairs(doc: dict, n_questions: int = QA_PER_DOC, max_retries: int = 3) -> list[dict]:
    """Generate Q/A pairs for a single document using the LLM."""
    user_prompt = make_user_prompt(
        doc['title'], doc['publisher'], doc['doc_markdown'], n_questions
    )
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.3,
                max_tokens=4096
            )
            
            raw = response.choices[0].message.content.strip()
            
            # Parse JSON — handle various formats
            # Sometimes LLM wraps in ```json ... ```
            if raw.startswith('```'):
                raw = re.sub(r'^```(?:json)?\s*', '', raw)
                raw = re.sub(r'\s*```$', '', raw)
            
            parsed = json.loads(raw)
            
            # Handle {"questions": [...]} wrapper (OpenAI json_object mode)
            if isinstance(parsed, dict):
                # Find the list in the dict values
                for v in parsed.values():
                    if isinstance(v, list):
                        parsed = v
                        break
            
            if not isinstance(parsed, list):
                raise ValueError(f"Expected list, got {type(parsed)}")
            
            # Validate structure
            valid = []
            for item in parsed:
                if 'question' in item and 'answer' in item:
                    valid.append({
                        'question': item['question'].strip(),
                        'answer': item['answer'].strip(),
                        'source_section': item.get('source_section', '').strip()
                    })
            
            if len(valid) >= n_questions - 2:  # Allow small margin
                return valid
            else:
                print(f"   ⚠️ Only {len(valid)} valid Q/A (expected {n_questions}), retrying...")
                
        except json.JSONDecodeError as e:
            print(f"   ⚠️ JSON parse error (attempt {attempt+1}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
        except Exception as e:
            print(f"   ⚠️ Error (attempt {attempt+1}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
    
    return []  # Failed after all retries


print(f"✅ Generation function ready")
print(f"   Prompt size: ~{len(SYSTEM_PROMPT)} chars system + doc content")

In [ ]:
# =============================================================================
# Cell 8 — TEST: Generate for 1 document
# =============================================================================

# Pick a medium-sized MATTE doc for testing
test_doc = next(d for d in documents if 'Fiche 1' in d['title'])
print(f"🧪 Testing on: [{test_doc['publisher']}] {test_doc['title']}")
print(f"   Size: {test_doc['char_count']:,} chars")

t0 = time.time()
test_qa = generate_qa_pairs(test_doc, n_questions=5)
elapsed = time.time() - t0

print(f"\n✅ Generated {len(test_qa)} Q/A pairs in {elapsed:.1f}s\n")
for i, qa in enumerate(test_qa, 1):
    print(f"--- Q{i} ---")
    print(f"Q: {qa['question']}")
    print(f"A: {qa['answer'][:200]}...")
    print(f"📌 Section: {qa['source_section']}")
    print()

In [ ]:
# =============================================================================
# Cell 9 — SAVE FUNCTION
# =============================================================================

def save_qa_pairs_to_db(qa_pairs: list[dict], doc: dict, goldset_name: str) -> int:
    """Save generated Q/A pairs to goldset_questions_v2.
    
    Stores:
    - question, gold_answer
    - gold_sources = short_id (for traceability & skip-existing)
    - source = 'synthetic'
    - theme = publisher
    - comment = left clean for future notes
    """
    saved = 0
    with psycopg.connect(dsn) as conn:
        with conn.cursor() as cur:
            for qa in qa_pairs:
                cur.execute('''
                    INSERT INTO goldset_questions_v2 
                    (question, gold_answer, gold_sources, goldset_name, source, theme, created_at)
                    VALUES (%s, %s, %s, %s, %s, %s, NOW())
                ''', (
                    qa['question'],
                    qa['answer'],
                    doc['short_id'],   # short_id for traceability
                    goldset_name,
                    'synthetic',
                    doc['publisher'],
                ))
                saved += 1
            conn.commit()
    return saved

print(f"✅ Save function ready")

In [ ]:
# =============================================================================
# Cell 10 — BATCH GENERATION (ALL DOCUMENTS)
# =============================================================================

print(f"🚀 Starting batch generation")
print(f"   Documents: {len(docs_to_process)}")
print(f"   Q/A per doc: {QA_PER_DOC}")
print(f"   Expected total: ~{len(docs_to_process) * QA_PER_DOC} questions")
print(f"   LLM: {LLM_PROVIDER}/{LLM_MODEL}")
print(f"   Save to DB: {SAVE_TO_DB}")
print()

all_results = []
total_saved = 0
errors = []
t0_batch = time.time()

for i, doc in enumerate(docs_to_process, 1):
    t0_doc = time.time()
    title_short = doc['title'][:70]
    print(f"[{i}/{len(docs_to_process)}] [{doc['publisher']}] {title_short}...")
    
    try:
        qa_pairs = generate_qa_pairs(doc, n_questions=QA_PER_DOC)
        elapsed_doc = time.time() - t0_doc
        
        if not qa_pairs:
            print(f"   ❌ No Q/A generated")
            errors.append((doc['doc_id'], doc['title'], "empty result"))
            continue
        
        # Save to DB
        if SAVE_TO_DB:
            n_saved = save_qa_pairs_to_db(qa_pairs, doc, GOLDSET_NAME)
            total_saved += n_saved
            print(f"   ✅ {len(qa_pairs)} Q/A → DB ({n_saved} saved) | {elapsed_doc:.1f}s")
        else:
            print(f"   ✅ {len(qa_pairs)} Q/A generated | {elapsed_doc:.1f}s")
        
        all_results.append({
            'doc_id': doc['doc_id'],
            'title': doc['title'],
            'publisher': doc['publisher'],
            'qa_pairs': qa_pairs,
            'count': len(qa_pairs)
        })
        
        # Rate limiting (be nice to the API)
        time.sleep(0.5)
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        errors.append((doc['doc_id'], doc['title'], str(e)))
        time.sleep(2)

elapsed_batch = time.time() - t0_batch
total_qa = sum(r['count'] for r in all_results)

print(f"\n{'='*60}")
print(f"🏁 BATCH COMPLETE")
print(f"   Documents processed: {len(all_results)}/{len(docs_to_process)}")
print(f"   Total Q/A generated: {total_qa}")
print(f"   Total saved to DB: {total_saved}")
print(f"   Errors: {len(errors)}")
print(f"   Elapsed: {elapsed_batch:.0f}s ({elapsed_batch/60:.1f}min)")
if errors:
    print(f"\n❌ Errors:")
    for doc_id, title, err in errors:
        print(f"   - {title[:60]}: {err[:80]}")

In [ ]:
# =============================================================================
# Cell 11 — VERIFY RESULTS
# =============================================================================

with psycopg.connect(dsn, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT count(*) as cnt,
                   count(DISTINCT gold_sources) as n_docs
            FROM goldset_questions_v2
            WHERE goldset_name = %s
        ''', (GOLDSET_NAME,))
        stats = cur.fetchone()
        
        cur.execute('''
            SELECT theme, count(*) as cnt
            FROM goldset_questions_v2
            WHERE goldset_name = %s
            GROUP BY theme
        ''', (GOLDSET_NAME,))
        by_theme = cur.fetchall()
        
        cur.execute('''
            SELECT question, gold_answer, gold_sources
            FROM goldset_questions_v2
            WHERE goldset_name = %s
            ORDER BY random()
            LIMIT 5
        ''', (GOLDSET_NAME,))
        samples = cur.fetchall()

print(f"📊 Goldset '{GOLDSET_NAME}' in DB:")
print(f"   {stats['cnt']} questions from {stats['n_docs']} documents")
print(f"\n   By publisher:")
for r in by_theme:
    print(f"     {r['theme']}: {r['cnt']} questions")

print(f"\n📋 Random samples:")
for s in samples:
    print(f"   Q: {s['question'][:100]}")
    print(f"   A: {s['gold_answer'][:120]}...")
    print(f"   Source: {s['gold_sources'][:80]}")
    print()

In [ ]:
# =============================================================================
# Cell 12 — COMPUTE EMBEDDINGS (for cached retrieval)
# =============================================================================

from sentence_transformers import SentenceTransformer
import numpy as np

print("🔧 Loading local BAAI/bge-m3 model...")
embed_model = SentenceTransformer("BAAI/bge-m3")
print("✅ Model loaded")

# Load questions without embeddings
with psycopg.connect(dsn, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute('''
            SELECT id, question 
            FROM goldset_questions_v2
            WHERE goldset_name = %s AND embedding_albert IS NULL
        ''', (GOLDSET_NAME,))
        to_embed = cur.fetchall()

print(f"📝 {len(to_embed)} questions need embeddings")

if to_embed:
    questions = [r['question'] for r in to_embed]
    ids = [r['id'] for r in to_embed]
    
    # Batch embed
    BATCH_SIZE = 32
    all_embeddings = []
    for start in range(0, len(questions), BATCH_SIZE):
        batch = questions[start:start+BATCH_SIZE]
        embs = embed_model.encode(batch, normalize_embeddings=True)
        all_embeddings.extend(embs)
        print(f"   Embedded {min(start+BATCH_SIZE, len(questions))}/{len(questions)}")
    
    # Save to DB
    with psycopg.connect(dsn) as conn:
        with conn.cursor() as cur:
            for qid, emb in zip(ids, all_embeddings):
                cur.execute(
                    'UPDATE goldset_questions_v2 SET embedding_albert = %s WHERE id = %s',
                    (emb.tolist(), qid)
                )
            conn.commit()
    print(f"✅ {len(all_embeddings)} embeddings saved to DB")
else:
    print("✅ All questions already have embeddings")

In [ ]:
# =============================================================================
# Cell 13 — EXPORT SUMMARY
# =============================================================================

with psycopg.connect(dsn, row_factory=dict_row) as conn:
    df = pd.read_sql('''
        SELECT id, question, gold_answer, gold_sources as short_id, theme
        FROM goldset_questions_v2
        WHERE goldset_name = %s
        ORDER BY theme, id
    ''', conn, params=(GOLDSET_NAME,))

print(f"📊 Final goldset: {len(df)} questions")
print(f"\n{df.groupby('theme').size()}")

# Save local CSV backup
csv_path = PROJECT_ROOT / f"notebooks/{GOLDSET_NAME}_export.csv"
df.to_csv(csv_path, index=False)
print(f"\n💾 Exported to: {csv_path}")